# KRONOS — Complete Project Flow

**KRONOS** is a faithfulness-first medical VQA system for chest X-rays. Given an image and a question, it combines:
- **Object detection** (YOLOv12s) to extract findings
- **Ontology-grounded symbolic reasoning** over a knowledge graph
- **Best-first tree search** guided by a deterministic verifier
- **A frozen VLM** (MedGemma 4B) as the action proposer

Every Tier-A answer carries a **graph path a human can check** — reasoning *through the graph*, not prose chain-of-thought.

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import yaml
import json
from pathlib import Path

# Shared style
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#0d1117",
    "text.color": "white",
    "axes.labelcolor": "white",
    "xtick.color": "white",
    "ytick.color": "white",
})

COLORS = {
    "blue": "#58a6ff",
    "green": "#3fb950",
    "orange": "#d29922",
    "red": "#f85149",
    "purple": "#bc8cff",
    "gray": "#8b949e",
    "cyan": "#39d353",
    "pink": "#f778ba",
}

DATA_DIR = Path("data/ontology")

## 1. High-Level Pipeline

The entire system flows through four stages: **Perception → Parsing → Tree Search → Result**.

```
Image + Question
      │
      ▼
┌────────────┐    ┌──────────┐    ┌─────────────┐    ┌──────────────┐
│ Perception │───▶│ Parsing  │───▶│ Tree Search │───▶│   Result     │
│ (Detector) │    │(Question)│    │   (Agent +  │    │(Tier A/B/∅)  │
│            │    │          │    │  Tools +    │    │              │
│            │    │          │    │  Verifier)  │    │              │
└────────────┘    └──────────┘    └─────────────┘    └──────────────┘
```

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.set_xlim(0, 16)
ax.set_ylim(0, 4)
ax.axis("off")
ax.set_title("KRONOS End-to-End Pipeline", fontsize=18, fontweight="bold", color="white", pad=15)

boxes = [
    (0.5,  1.0, 2.5, 2.0, "① Perception\n(YOLOv12s)", COLORS["blue"]),
    (4.0,  1.0, 2.5, 2.0, "② Parsing\n(Question→Query)", COLORS["orange"]),
    (7.5,  1.0, 4.0, 2.0, "③ Tree Search\n(Agent + Tools + Verifier)", COLORS["green"]),
    (12.5, 1.0, 2.5, 2.0, "④ Result\n(Tier A / B / ∅)", COLORS["purple"]),
]

for x, y, w, h, label, color in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
                                    facecolor=color + "22", edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center",
            fontsize=11, fontweight="bold", color=color)

arrows = [(3.0, 2.0, 0.8, 0), (6.5, 2.0, 0.8, 0), (11.5, 2.0, 0.8, 0)]
for x, y, dx, dy in arrows:
    ax.annotate("", xy=(x + dx, y), xytext=(x, y),
                arrowprops=dict(arrowstyle="-|>", color="white", lw=2))

# Input labels
ax.text(1.75, 3.5, "🖼️ CXR Image", ha="center", fontsize=10, color=COLORS["gray"])
ax.text(5.25, 3.5, "❓ Question", ha="center", fontsize=10, color=COLORS["gray"])
ax.annotate("", xy=(1.75, 3.0), xytext=(1.75, 3.35),
            arrowprops=dict(arrowstyle="-|>", color=COLORS["gray"], lw=1.5))
ax.annotate("", xy=(5.25, 3.0), xytext=(5.25, 3.35),
            arrowprops=dict(arrowstyle="-|>", color=COLORS["gray"], lw=1.5))

plt.tight_layout()
plt.show()

---
## 2. Perception — Extracting Facts from the Image

**File:** `src/perception/detector.py` (real) · `src/perception/oracle.py` (ground-truth)

The detector takes a raw CXR image and produces a list of **PerceptualFact** objects — the grounded evidence the rest of the system reasons over.

| Field | Type | Example |
|-------|------|---------|
| `concept` | str | `"Cardiomegaly"` |
| `bbox` | (x1,y1,x2,y2) | `(800, 900, 1500, 1800)` |
| `laterality` | str | `"midline"` |
| `conf` | float | `0.92` |

**Two paths:**
- **Real:** YOLOv12s runs inference → each detection becomes a PerceptualFact
- **Oracle:** Ground-truth from VinDr `train.csv` → multi-radiologist bboxes averaged, conf=1.0

In [ ]:
from src.ontology.dag import OntologyDAG
from src.perception.oracle import PerceptionOracle
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

dag = OntologyDAG(
    str(DATA_DIR / "dag.yaml"),
    exclusion_path=str(DATA_DIR / "exclusion_lists.yaml"),
    zones_path=str(DATA_DIR / "anatomy_zones.yaml"),
)

oracle = PerceptionOracle(
    "data/vindr_cxr_vqa/train.csv",
    dag=dag
)

IMAGE_ID = "9a5094b2563a1ef3ff50dc5c7ff71345"
IMAGE_PATH = f"data/vindr_cxr_vqa/train/{IMAGE_ID}.png"

real_facts = oracle.get_facts(IMAGE_ID)

print(f"Image: {IMAGE_ID}")
print(
    f"Oracle extracted {len(real_facts)} findings "
    f"(from {len(oracle._facts.get(IMAGE_ID, []))} annotations):\n"
)

for f in real_facts:
    print(
        f"  {f.concept:20s} "
        f"bbox=({f.bbox[0]:.0f}, {f.bbox[1]:.0f}, "
        f"{f.bbox[2]:.0f}, {f.bbox[3]:.0f}) "
        f"lat={f.laterality} conf={f.conf}"
    )

# Load and display the REAL chest X-ray with bounding boxes
image = Image.open(IMAGE_PATH).convert("RGB")

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(image, cmap="gray")

ax.set_title(
    f"Real CXR: {IMAGE_ID}\n(Oracle facts overlaid)",
    fontsize=12,
    fontweight="bold",
    pad=10,
)

fact_colors = ["red", "blue", "orange", "green"]

for fact, color in zip(real_facts, fact_colors):

    x1, y1, x2, y2 = fact.bbox

    # Oracle scales coordinates to 512×512
    scale_x = image.size[0] / 512.0
    scale_y = image.size[1] / 512.0

    rx1 = x1 * scale_x
    ry1 = y1 * scale_y
    rx2 = x2 * scale_x
    ry2 = y2 * scale_y

    rect = mpatches.Rectangle(
        (rx1, ry1),
        rx2 - rx1,
        ry2 - ry1,
        linewidth=2,
        edgecolor=color,
        facecolor=color,
        alpha=0.2,
    )

    ax.add_patch(rect)

    ax.text(
        rx1,
        ry1 - 5,
        f"{fact.concept} ({fact.laterality})",
        fontsize=8,
        fontweight="bold",
        color=color,
        va="bottom",
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor="black",
            edgecolor=color,
            alpha=0.7,
        ),
    )

ax.axis("off")
plt.tight_layout()
plt.show()

### Laterality Logic

Laterality is derived from the bbox center position using **PA CXR convention** (image left = patient right):

```
           ←── image width ──→
    0.0          0.5          1.0
     │    RIGHT   │   LEFT     │
     │  (patient) │ (patient)  │
     │            │            │
     │   center < 0.5 → right │
     │   center ≥ 0.5 → left  │
     │   wide box → bilateral │
     │   narrow center → midline │
```

In [ ]:
# Visualize the laterality decision boundaries
fig, ax = plt.subplots(figsize=(10, 3))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect(0.3)
ax.set_title("Laterality Decision Regions (PA Convention)", fontsize=13, fontweight="bold", pad=10)

ax.axvspan(0.0, 0.38, alpha=0.15, color=COLORS["blue"], label="RIGHT (patient)")
ax.axvspan(0.38, 0.62, alpha=0.15, color=COLORS["gray"], label="MIDLINE zone")
ax.axvspan(0.62, 1.0, alpha=0.15, color=COLORS["red"], label="LEFT (patient)")
ax.axvline(0.5, color="white", linestyle="--", linewidth=1, alpha=0.5, label="Image midline")

ax.text(0.19, 0.5, "RIGHT\n(patient)", ha="center", va="center", fontsize=12, fontweight="bold", color=COLORS["blue"])
ax.text(0.50, 0.5, "MIDLINE", ha="center", va="center", fontsize=11, fontweight="bold", color=COLORS["gray"])
ax.text(0.81, 0.5, "LEFT\n(patient)", ha="center", va="center", fontsize=12, fontweight="bold", color=COLORS["red"])

ax.text(0.50, 0.05, "center_x = bbox_center / image_width", ha="center", fontsize=9, color=COLORS["gray"], style="italic")

ax.set_xlabel("Normalized x-position (center_x)", fontsize=10)
ax.set_yticks([])
ax.legend(loc="upper center", ncol=4, fontsize=8, framealpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Question Parsing — Classifying the Question

**File:** `src/question/parser.py`

The parser converts a raw question string into a structured **Query** object. It uses a **two-tier** approach:

- **Tier 1 (Rule-based, conf=1.0):** Pattern-matching in priority order
- **Tier 2 (LLM fallback, conf=0.5):** Groq Llama 3.3 70B classifies if no rule fires

### Priority order of rules:

| Priority | Pattern | Question Type | Example |
|----------|---------|---------------|---------|
| 1 | Negation cue (`no, not, without, clear of, ...`) | `negation` | "Are the lungs clear?" |
| 2 | `"how many"` | `counting` | "How many findings?" |
| 3 | `"where is"` | `relational` | "Where is the effusion?" |
| 4 | `"which side"` | `relational` | "Which side shows it?" |
| 5 | `"what abnormality/finding"` | `open` | "What abnormality is visible?" |
| 6 | `"is there"` / `"does...show"` | `existential` | "Is there Cardiomegaly?" |
| 7 | Nothing matched | LLM or `relational` fallback | anything else |

**Key detail:** Negation is checked **first** to avoid misclassifying "Are the lungs clear?" as existential.

In [ ]:
# Parse REAL VQA questions from vqa.json for our sample image
from src.question.parser import QuestionParser

VINDR_FINDINGS = [
    "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
    "Consolidation", "ILD", "Infiltration", "Lung Opacity",
    "Nodule/Mass", "Other lesion", "Pleural effusion",
    "Pleural thickening", "Pneumothorax", "Pulmonary fibrosis",
]
parser = QuestionParser(finding_vocab=VINDR_FINDINGS)

# Load real VQA questions for our image
with open("data/vindr_cxr_vqa/vqa.json", encoding="utf-8") as f:
    vqa_data = json.load(f)

real_questions = []
for entry in vqa_data:
    if entry["image_id"] == IMAGE_ID:
        for qa in entry["vqa"]:
            real_questions.append(qa["question"])
        break

# Also add a negation question to show all types
real_questions.append("Are the lungs clear of Consolidation?")
real_questions.append("Which side shows the abnormality?")

print(f"Parsing {len(real_questions)} questions for image {IMAGE_ID}:\n")
print(f"{'Question':<50} {'Type':<14} {'Target':<20} {'Conf':<6} {'Tier'}")
print("─" * 110)
for q in real_questions:
    query = parser.parse(q)
    target = query.target or "(none)"
    print(f"{q:<50} {query.type:<14} {target:<20} {query.parse_confidence:<6.1f} {query.parser_tier}")

In [ ]:
# Visualize the parser decision tree
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")
ax.set_title("Question Parser — Decision Flow", fontsize=15, fontweight="bold", pad=12)

def draw_box(ax, x, y, w, h, text, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12",
                                    facecolor=color + "22", edgecolor=color, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize,
            fontweight="bold", color=color)

def draw_arrow(ax, x1, y1, x2, y2, label="", color="white"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.5))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.15, my, label, fontsize=7, color=COLORS["gray"], style="italic")

# Input
draw_box(ax, 5.5, 7.0, 3.0, 0.7, "Raw Question", COLORS["gray"], 11)

# Decision nodes (diamonds as boxes)
checks = [
    (5.5, 5.8, "Negation cue?",   COLORS["red"],    "negation"),
    (5.5, 4.6, '"how many"?',     COLORS["orange"],  "counting"),
    (5.5, 3.4, '"where is"?',     COLORS["blue"],    "relational"),
    (5.5, 2.2, '"is there"?',     COLORS["green"],   "existential"),
    (5.5, 1.0, "LLM fallback",    COLORS["purple"],  "llm / relational"),
]

draw_arrow(ax, 7.0, 7.0, 7.0, 6.6)

for i, (x, y, label, color, result) in enumerate(checks):
    draw_box(ax, x, y, 3.0, 0.6, label, color, 10)
    # "yes" arrow to the right → result
    draw_box(ax, 10.0, y, 2.8, 0.6, f"→ {result}", color, 9)
    draw_arrow(ax, 8.5, y + 0.3, 10.0, y + 0.3, "yes", color)
    # "no" arrow down to next check
    if i < len(checks) - 1:
        next_y = checks[i+1][1]
        draw_arrow(ax, 7.0, y, 7.0, next_y + 0.6, "no")

draw_arrow(ax, 7.0, 6.6, 7.0, checks[0][1] + 0.6)

ax.text(1.0, 7.2, "Priority order matters!\nNegation checked first\nto avoid misclassifying\n'Are lungs clear?' as\nexistential.",
        fontsize=8, color=COLORS["gray"], style="italic", va="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#1a1a2e", edgecolor=COLORS["gray"], alpha=0.7))

plt.tight_layout()
plt.show()

---
## 4. Ontology DAG — The Knowledge Graph

**File:** `src/ontology/dag.py` · **Data:** `data/ontology/dag.yaml`

The `OntologyDAG` is a hand-curated NetworkX DiGraph with ~33 nodes. It serves **five reasoning roles**:

| Role | Method | What it does |
|------|--------|-------------|
| **1. Subsumption** | `reachable_is_a(a, b)` | Walks is-a edges upward: "is consolidation a pulmonary abnormality?" |
| **2. Disjointness** | `disjoint(a, b)` | Checks mutual exclusion: pneumothorax ⊥ pleural effusion on same region |
| **3. Anatomy mapping** | `anatomy_of(bbox, w, h)` | Maps bbox → anatomy zone via IoU |
| **4. Laterality** | `compose_laterality(bbox, w, h)` | Maps bbox position → left/right/bilateral/midline |
| **5. Causal graph** | `causal_neighbors`, `find_causal_path` | Multi-hop reasoning over RGO may_cause edges |

In [ ]:
# Visualize the is-a ontology DAG (dag already loaded above from real data)
# Build the is-a subgraph for visualization
isa_graph = nx.DiGraph()
for u, v, d in dag.graph.edges(data=True):
    if d.get("relation") == "is-a":
        src_name = dag.graph.nodes[u].get("name", u)
        tgt_name = dag.graph.nodes[v].get("name", v)
        isa_graph.add_edge(src_name, tgt_name)

node_colors = []
for n in isa_graph.nodes():
    if n == "Abnormality":
        node_colors.append(COLORS["red"])
    elif n in ("Cardiac abnormality", "Pulmonary abnormality", "Pleural abnormality", "Vascular abnormality"):
        node_colors.append(COLORS["orange"])
    elif n in ("Airspace abnormality", "Interstitial abnormality"):
        node_colors.append(COLORS["purple"])
    else:
        node_colors.append(COLORS["green"])

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_title("Ontology DAG — is-a Hierarchy (14 VinDr Findings)", fontsize=15, fontweight="bold", pad=15)

pos = {}
pos["Abnormality"] = (0, 4)
pos["Cardiac abnormality"] = (-6, 3)
pos["Pulmonary abnormality"] = (-1, 3)
pos["Pleural abnormality"] = (5, 3)
pos["Vascular abnormality"] = (9, 3)
pos["Airspace abnormality"] = (-4, 2)
pos["Interstitial abnormality"] = (1, 2)
pos["Cardiomegaly"] = (-6, 1)
pos["Consolidation"] = (-7, 1)
pos["Infiltration"] = (-5.5, 1)
pos["Lung Opacity"] = (-4, 1)
pos["Atelectasis"] = (-2.5, 1)
pos["ILD"] = (0.5, 1)
pos["Pulmonary fibrosis"] = (2, 1)
pos["Nodule/Mass"] = (-1, 1)
pos["Pleural effusion"] = (4, 1)
pos["Pleural thickening"] = (5.5, 1)
pos["Pneumothorax"] = (7, 1)
pos["Aortic enlargement"] = (9, 1)
pos["Calcification"] = (3, 4)
pos["Other lesion"] = (5, 4.5)

nx.draw_networkx_nodes(isa_graph, pos, ax=ax, node_color=node_colors, node_size=1800, alpha=0.85)
nx.draw_networkx_edges(isa_graph, pos, ax=ax, edge_color=COLORS["gray"], arrows=True,
                       arrowstyle="-|>", arrowsize=15, width=1.5, alpha=0.6)
nx.draw_networkx_labels(isa_graph, pos, ax=ax, font_size=7, font_weight="bold", font_color="white")

legend_items = [
    mpatches.Patch(color=COLORS["red"], label="Root"),
    mpatches.Patch(color=COLORS["orange"], label="Organ system"),
    mpatches.Patch(color=COLORS["purple"], label="Subgroup"),
    mpatches.Patch(color=COLORS["green"], label="VinDr finding (leaf)"),
]
ax.legend(handles=legend_items, loc="lower left", fontsize=9, framealpha=0.3)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"DAG stats: {dag.graph.number_of_nodes()} nodes, {dag.graph.number_of_edges()} edges")
print(f"Edge types: { {d['relation'] for _, _, d in dag.graph.edges(data=True)} }")

In [ ]:
# Demo: is-a reasoning — walk upward from a leaf finding
print("=== Role 1: Subsumption (is-a) ===\n")

queries = [
    ("consolidation", "airspace_abnormality"),
    ("consolidation", "pulmonary_abnormality"),
    ("consolidation", "abnormality"),
    ("cardiomegaly", "pulmonary_abnormality"),  # should fail — different branch
]
for node, target in queries:
    path = dag.reachable_is_a(node, target)
    status = " → ".join(path) if path else "None (not reachable)"
    symbol = "✓" if path else "✗"
    print(f"  {symbol}  is_a({node}, {target})")
    print(f"       path: {status}\n")

print("=== Role 2: Disjointness ===\n")
pairs = [("pneumothorax", "pleural_effusion"), ("cardiomegaly", "consolidation")]
for a, b in pairs:
    result = dag.disjoint(a, b)
    print(f"  disjoint({a}, {b}) = {result}")

print("\n=== Role 3: Anatomy Mapping ===\n")
test_bboxes = [
    ((800, 900, 1500, 1800), "midline finding"),
    ((50, 500, 400, 1200),   "right-side finding"),
    ((1800, 500, 2200, 1200), "left-side finding"),
]
for bbox, desc in test_bboxes:
    zone = dag.anatomy_of(bbox, 2304, 2880)
    lat = dag.compose_laterality(bbox, 2304, 2880)
    print(f"  {desc}: bbox={bbox}")
    print(f"       anatomy_of → {zone},  compose_laterality → {lat}\n")

### Role 5: Causal Graph (RGO may_cause)

**Data:** `data/ontology/causal_kg.yaml` — ~160 disorder/observation nodes, ~400 `may_cause` edges from the Radiology Gamuts Ontology.

The 14 VinDr findings are mapped to RGO concepts via a `seeds` dictionary. This enables **multi-hop causal reasoning**: "What disorder could cause both Cardiomegaly and Pleural effusion?"

In [ ]:
# Visualize the causal neighborhood of Cardiomegaly
target_finding = "Cardiomegaly"
causes = dag.causal_neighbors(target_finding, "caused_by")[:12]
effects = dag.causal_neighbors(target_finding, "causes")[:6]

G = nx.DiGraph()
G.add_node(target_finding, role="seed")
for c in causes:
    G.add_node(c, role="cause")
    G.add_edge(c, target_finding)
for e in effects:
    G.add_node(e, role="effect")
    G.add_edge(target_finding, e)

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_title(f"Causal Neighborhood of '{target_finding}' (RGO may_cause)", fontsize=14, fontweight="bold", pad=12)

pos = nx.spring_layout(G, k=2.5, seed=42)
node_c = []
for n in G.nodes():
    role = G.nodes[n].get("role")
    if role == "seed":
        node_c.append(COLORS["red"])
    elif role == "cause":
        node_c.append(COLORS["blue"])
    else:
        node_c.append(COLORS["orange"])

nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_c, node_size=2000, alpha=0.85)
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=COLORS["gray"], arrows=True,
                       arrowstyle="-|>", arrowsize=15, width=1.5, alpha=0.6,
                       connectionstyle="arc3,rad=0.05")
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_weight="bold", font_color="white")

legend_items = [
    mpatches.Patch(color=COLORS["blue"], label=f"Causes of {target_finding}"),
    mpatches.Patch(color=COLORS["red"], label=target_finding),
    mpatches.Patch(color=COLORS["orange"], label=f"Effects of {target_finding}"),
]
ax.legend(handles=legend_items, loc="lower left", fontsize=9, framealpha=0.3)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Causes of {target_finding} ({len(causes)} shown): {causes}")
print(f"Effects of {target_finding} ({len(effects)} shown): {effects}")

In [ ]:
# Demo: multi-hop causal paths + shared-cause queries
print("=== Causal Paths ===\n")
path_queries = [
    ("tuberculosis", "Pleural effusion"),
    ("sarcoidosis", "Cardiomegaly"),
    ("tuberculosis", "Pneumothorax"),
]
for src, tgt in path_queries:
    path = dag.find_causal_path(src, tgt)
    if path:
        print(f"  {src} → {tgt}:")
        print(f"    {' → '.join(path)}\n")
    else:
        print(f"  {src} → {tgt}: no path\n")

print("=== Shared Causes (multi-hop QA core) ===\n")
finding_pairs = [
    ("Cardiomegaly", "Pleural effusion"),
    ("Pneumothorax", "Pleural thickening"),
    ("Cardiomegaly", "Pneumothorax"),
]
for a, b in finding_pairs:
    shared = dag.common_causes(a, b)
    print(f"  Common causes of ({a}, {b}):")
    if shared:
        print(f"    {shared[:8]}{'...' if len(shared) > 8 else ''}")
    else:
        print(f"    (none)")
    print()

---
## 5. Tool Layer — Executing Actions

**Files:** `src/tools/dispatch.py` · `src/tools/symbolic.py` · `src/tools/visual.py` · `src/retrieval/tool.py`

The tool layer converts an `Action` into an `Observation`. The dispatcher routes by type:

```
Action
  │
  ├── tool == "retrieve"  ──→  RAG retrieval (BiomedCLIP + FAISS)
  ├── kind == "visual"    ──→  Visual tools (need image + model)
  │     ├── inspect(bbox)       → VLM describes a region
  │     ├── re_detect(bbox)     → detector re-runs on a crop
  │     └── compare(bbox1,bbox2)→ VLM compares two regions
  └── kind == "symbolic"  ──→  Symbolic tools (deterministic, no model)
        ├── is_a(node, target)
        ├── disjoint(a, b)
        ├── anatomy_of(bbox)
        ├── compose_laterality(bbox)
        ├── get_exclusion_list(name)
        ├── neighbors(node, dir)
        └── find_path(src, tgt)
```

**Key design:** Symbolic tools are **fast and deterministic** — the agent is instructed to prefer them. Visual tools are only used when symbolic evidence is insufficient.

In [ ]:
# Demo: execute symbolic tools directly using real_facts from the oracle
from src.contracts import Action, Observation
from src.tools.symbolic import run_tool as run_symbolic

print("=== Symbolic Tool Execution Demo ===")
print(f"Using {len(real_facts)} real facts from image {IMAGE_ID}\n")

# Tool 1: is_a — does Cardiomegaly witness cardiac_abnormality?
action = Action(tool="is_a", args={"node": "cardiomegaly", "target": "cardiac_abnormality"})
obs = run_symbolic(action, real_facts, dag, (2304, 2880))
print(f"  is_a(cardiomegaly, cardiac_abnormality)")
print(f"    \u2192 ok={obs.ok}, result={obs.result}\n")

# Tool 2: disjoint — are Pneumothorax and Pleural effusion mutually exclusive?
action = Action(tool="disjoint", args={"a": "pneumothorax", "b": "pleural_effusion"})
obs = run_symbolic(action, real_facts, dag, (2304, 2880))
print(f"  disjoint(pneumothorax, pleural_effusion)")
print(f"    \u2192 ok={obs.ok}, result={obs.result}\n")

# Tool 3: anatomy_of — which zone does the Cardiomegaly bbox fall in?
cardio_bbox = real_facts[0].bbox  # Cardiomegaly
action = Action(tool="anatomy_of", args={"bbox": list(cardio_bbox)})
obs = run_symbolic(action, real_facts, dag, (512, 512))
print(f"  anatomy_of(bbox={[round(x) for x in cardio_bbox]})  # Cardiomegaly")
print(f"    \u2192 ok={obs.ok}, result={obs.result}\n")

# Tool 4: compose_laterality — what side is the Pleural effusion on?
pe_bbox = real_facts[1].bbox  # Pleural effusion
action = Action(tool="compose_laterality", args={"bbox": list(pe_bbox)})
obs = run_symbolic(action, real_facts, dag, (512, 512))
print(f"  compose_laterality(bbox={[round(x) for x in pe_bbox]})  # Pleural effusion")
print(f"    \u2192 ok={obs.ok}, result={obs.result}\n")

# Tool 5: get_exclusion_list — what findings would rule out Cardiomegaly?
action = Action(tool="get_exclusion_list", args={"name": "Cardiomegaly"})
obs = run_symbolic(action, real_facts, dag, (512, 512))
print(f"  get_exclusion_list('Cardiomegaly')")
print(f"    \u2192 ok={obs.ok}, result={obs.result}\n")

# Tool 6: neighbors — what disorders can cause Cardiomegaly?
action = Action(tool="neighbors", args={"node": "Cardiomegaly", "direction": "caused_by"})
obs = run_symbolic(action, real_facts, dag, (512, 512))
print(f"  neighbors('Cardiomegaly', 'caused_by')")
print(f"    \u2192 ok={obs.ok}, result={obs.result[:5]}... ({len(obs.result)} total)\n")

# Tool 7: find_path — causal chain from sarcoidosis to Cardiomegaly
action = Action(tool="find_path", args={"source": "sarcoidosis", "target": "Cardiomegaly"})
obs = run_symbolic(action, real_facts, dag, (512, 512))
print(f"  find_path('sarcoidosis', 'Cardiomegaly')")
print(f"    \u2192 ok={obs.ok}, result={obs.result}")


---
## 6. Agent — Proposing Actions

**Files:** `src/agent/base.py` · `src/agent/mock.py` · `src/agent/medgemma.py` · `src/agent/prompt.py`

The agent receives the current `TreeNode` (facts + history) and the `Query`, then proposes up to `k` next actions — or an answer string if it has enough evidence.

**Two implementations:**
- **MockAgent** — deterministic scripted logic per question type (for testing)
- **MedGemmaAgent** — frozen MedGemma 4B, prompt-only, greedy decoding (for real inference)

### MockAgent strategy per question type:

| Type | Strategy |
|------|----------|
| existential | For each fact → `is_a(fact, target)`. If all checked → answer "No" |
| negation | First `get_exclusion_list(target)`, then `is_a` for each item. If clear → answer |
| relational | Find matching fact → `anatomy_of(bbox)` or `compose_laterality(bbox)` |
| counting | Count distinct fact concepts → answer directly |
| open | List all fact names → answer directly |

In [ ]:
# Demo: MedGemmaAgent proposing actions for different question types
from src.agent.medgemma import MedGemmaAgent
from src.contracts import TreeNode, Query
from PIL import Image

image = Image.open(IMAGE_PATH).convert("RGB")
agent = MedGemmaAgent(model_path="weights/medgemma-4b-it", quantize=True, load_model=True, image=image)

demos = [
    ("existential", "Is there Cardiomegaly?",            "Cardiomegaly", {}),
    ("negation",    "Are the lungs clear of Consolidation?", "Consolidation", {}),
    ("relational",  "Where is the Pleural effusion?",     "Pleural effusion", {"attr": "location"}),
    ("counting",    "How many findings are there?",       None, {}),
    ("open",        "What abnormality is visible?",       None, {}),
]

for qtype, raw_q, target, constraints in demos:
    query = Query(type=qtype, target=target, constraints=constraints,
                  raw_question=raw_q, parse_confidence=1.0, parser_tier="rule")
    node = TreeNode(state_facts=list(real_facts), history=[])
    proposals = agent.propose_actions(node, query, k=3)

    print(f"Q: \"{raw_q}\"  (type={qtype})")
    for p in proposals:
        if isinstance(p, str):
            print(f"  → Answer: \"{p}\"")
        else:
            print(f"  → Action: {p.tool}({p.args})")
    print()


### MedGemma Prompt Structure

When using the real MedGemma agent, `build_prompt()` constructs a structured prompt with these sections:

```
┌──────────────────────────────────────────┐
│  SYSTEM INSTRUCTION                      │
│  (strategy per question type, rules)     │
├──────────────────────────────────────────┤
│  Question: "Is there Cardiomegaly?"      │
│  Question type: existential              │
│  Target: Cardiomegaly                    │
├──────────────────────────────────────────┤
│  Evidence facts:                         │
│    - Cardiomegaly (conf=0.92, bbox=...)  │
│    - Pleural effusion (conf=0.87, ...)   │
├──────────────────────────────────────────┤
│  Actions taken so far:                   │
│    Action: is_a({node: ..., target: ...})│
│    Result: [...] (ok=True)               │
├──────────────────────────────────────────┤
│  Reflection: (if previous answer failed) │
│    "No detected finding is-a target..."  │
├──────────────────────────────────────────┤
│  Available tools: (11 tools described)   │
├──────────────────────────────────────────┤
│  Output format:                          │
│    JSON array of actions, or Answer[...] │
└──────────────────────────────────────────┘
```

The model's raw output is parsed by `parse_output()`:
- `Answer[Yes]` → returns `["Yes"]`
- `[{"tool": "is_a", "args": {...}}]` → returns `[Action(...)]`

In [ ]:
# Show what the actual prompt looks like for one example
from src.agent.prompt import build_prompt, parse_output

query = Query(type="existential", target="Cardiomegaly", constraints={},
              raw_question="Is there Cardiomegaly?", parse_confidence=1.0, parser_tier="rule")
node = TreeNode(state_facts=list(real_facts), history=[])

prompt = build_prompt(node, query, k=3)
print("=" * 70)
print("PROMPT SENT TO MEDGEMMA (first 1500 chars):")
print("=" * 70)
print(prompt[:1500])
print(f"\n... ({len(prompt)} total characters)")

# Show parse_output on different model outputs
print("\n" + "=" * 70)
print("parse_output() examples:")
print("=" * 70)
test_outputs = [
    'Answer[Yes]',
    '[{"tool": "is_a", "args": {"node": "cardiomegaly", "target": "cardiac_abnormality"}}]',
    'I think the answer is Answer[No] based on the evidence.',
    'some junk with no valid output',
]
for raw in test_outputs:
    result = parse_output(raw)
    print(f"  Input:  {raw[:70]}")
    print(f"  Output: {result}\n")

---
## 7. Verifier — Scoring and Gating

**File:** `src/engine/verifier.py`

The verifier has **two roles** — it never uses an LLM, it is purely deterministic:

### 7a. `closure_progress(node, query, dag)` → float (0.0–1.0)

The **search value function** — tells tree search how promising a node is.

### 7b. `verify(node, query, dag)` → SearchResult

The **terminal gate** — when an answer is proposed, assigns it a tier:

| Tier | Meaning | When assigned |
|------|---------|---------------|
| **A** | Verified, graph-backed | is-a witness found, or closed-world absence confirmed, or anatomy resolved |
| **B** | Advisory, unverified | Model answered but the graph path is incomplete |
| **ABSTAIN** | Insufficient evidence | Nothing to say |

### 7c. `explain(node, query, dag)` → str

When an answer is rejected, generates a **reflection** message fed back to the agent on retry.

In [ ]:
# Visualize the verifier scoring logic per question type
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("closure_progress() — Verifier Scoring by Question Type", fontsize=14, fontweight="bold", y=1.02)

verifier_data = {
    "existential": [
        ("Witness found\n(is_a path)", 1.0, COLORS["green"]),
        ("Direct fact\nmatch", 1.0, COLORS["green"]),
        ("Target is\nknown concept", 0.2, COLORS["orange"]),
        ("No info", 0.1, COLORS["gray"]),
    ],
    "negation": [
        ("Exclusion clear\n(all checked)", 1.0, COLORS["green"]),
        ("Exclusion list\nnot yet fetched", 0.1, COLORS["orange"]),
        ("Fact matches\nexclusion list", 0.0, COLORS["red"]),
    ],
    "relational": [
        ("anatomy_of or\nlaterality OK", 1.0, COLORS["green"]),
        ("Not yet\nresolved", 0.2, COLORS["orange"]),
    ],
    "counting": [
        ("Always\n(trivially\nderivable)", 1.0, COLORS["green"]),
    ],
}

for ax, (qtype, bars) in zip(axes, verifier_data.items()):
    labels = [b[0] for b in bars]
    values = [b[1] for b in bars]
    colors = [b[2] for b in bars]
    x = range(len(bars))
    ax.bar(x, values, color=colors, alpha=0.8, width=0.6, edgecolor="white", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=7, ha="center")
    ax.set_ylim(0, 1.15)
    ax.set_title(qtype, fontsize=11, fontweight="bold", color=COLORS["blue"])
    ax.set_ylabel("Score" if ax == axes[0] else "", fontsize=9)
    for i, v in enumerate(values):
        ax.text(i, v + 0.03, f"{v:.1f}", ha="center", fontsize=9, fontweight="bold", color="white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(COLORS["gray"])
    ax.spines["bottom"].set_color(COLORS["gray"])

plt.tight_layout()
plt.show()

In [ ]:
# Demo: verifier in action on concrete tree nodes
from src.engine.verifier import closure_progress, verify, explain

print("=== Verifier Demo ===\n")

# Scenario 1: existential question — direct match in facts
query_ex = Query(type="existential", target="Cardiomegaly", constraints={},
                 raw_question="Is there Cardiomegaly?", parse_confidence=1.0, parser_tier="rule")
node_ex = TreeNode(state_facts=list(real_facts), history=[], answer="Yes")

score = closure_progress(node_ex, query_ex, dag)
result = verify(node_ex, query_ex, dag)
print(f"Scenario 1: 'Is there Cardiomegaly?' (Cardiomegaly IS in facts)")
print(f"  closure_progress = {score:.1f}")
print(f"  verify → tier={result.tier}, answer='{result.answer}', conf={result.conf:.2f}\n")

# Scenario 2: existential — target NOT in facts
query_miss = Query(type="existential", target="Pneumothorax", constraints={},
                   raw_question="Is there Pneumothorax?", parse_confidence=1.0, parser_tier="rule")
node_miss = TreeNode(state_facts=list(real_facts), history=[], answer="No")

score2 = closure_progress(node_miss, query_miss, dag)
result2 = verify(node_miss, query_miss, dag)
print(f"Scenario 2: 'Is there Pneumothorax?' (Pneumothorax NOT in facts)")
print(f"  closure_progress = {score2:.1f}")
print(f"  verify → tier={result2.tier}, answer='{result2.answer}', conf={result2.conf:.2f}\n")

# Scenario 3: failed answer → explain generates reflection
node_fail = TreeNode(state_facts=list(real_facts), history=[], answer="Maybe")
explanation = explain(node_fail, query_miss, dag)
print(f"Scenario 3: explain() on failed Pneumothorax answer:")
print(f"  \"{explanation[:120]}...\"")

---
## 8. Tree Search — The Core Reasoning Loop

**File:** `src/search/tree_search.py`

This is where everything comes together. The search algorithm:

1. Creates a **root node** with the detected facts
2. Maintains a **frontier** (priority queue sorted by verifier score)
3. At each step, expands the **best node** by asking the agent for proposals
4. **Actions** are executed → new child nodes scored → added to frontier
5. **Answers** are verified → Tier A returned immediately, Tier B saved as fallback
6. If answer **rejected** → `explain()` generates a reflection → node re-queued for retry

### Key properties:
- **Best-first:** always expands the most promising node
- **Budget-bounded:** max 20 expansions (default)
- **Implicit backtracking:** all live nodes in frontier, no explicit backtrack
- **Reflection loop:** rejected answers feed back to the agent via `node.reflection`

### 8a. Reflect/Explain Loop — Self-Correction Without Fine-Tuning

When the agent proposes an answer but the verifier rejects it (Tier B or ABSTAIN), the search doesn't just discard the attempt — it **learns from the failure**:

```
Agent proposes Answer["Yes"]
        │
        ▼
  verify() → Tier B or ABSTAIN (rejected)
        │
        ▼
  explain(node, query, dag) → deterministic reason string
        │
        ▼
  Re-queue the SAME node with node.reflection = reason
        │
        ▼
  Agent sees reflection in its next prompt → revises strategy
```

**How it works (`tree_search.py` lines 49–54):**

1. The verifier rejects the agent's answer (not Tier A)
2. `explain()` generates a **deterministic, query-type-specific** reason — no LLM involved
3. The original node (not the child) is copied with `reflection` set to the reason string
4. This retry node goes back on the frontier
5. A **once-per-node guard** (`reflected` flag) prevents infinite retry loops

**How the agent sees it (`prompt.py` line 81–82):**

When `build_prompt()` constructs the next prompt and `node.reflection` is non-empty, it appends:

```
Reflection from failed branch: <reason string>
```

This gives the frozen LLM a **corrective signal** without any weight updates — the prompt itself carries the feedback.

**Example `explain()` outputs by question type:**

| Question type | Example reflection |
|---|---|
| **existential** | "No detected finding is-a 'Cardiomegaly'... Try re_detect near a suspected region then is_a on new findings" |
| **negation** | "The exclusion list for 'Pneumothorax' was not fetched. Call get_exclusion_list('Pneumothorax') first" |
| **relational** | "Location is unresolved. Call anatomy_of(bbox) or compose_laterality(bbox) on the target finding's bbox" |
| **other** | "Insufficient verified evidence; gather more before answering" |

This is how KRONOS achieves **self-correction with a frozen model**: the verifier diagnoses the failure, and the search re-presents the problem with guidance.

In [ ]:
# Visualize the tree search algorithm as a flowchart
fig, ax = plt.subplots(figsize=(14, 14))
ax.set_xlim(0, 14)
ax.set_ylim(0, 14)
ax.axis("off")
ax.set_title("Tree Search Algorithm — Step by Step", fontsize=16, fontweight="bold", pad=12)

def box(ax, x, y, w, h, text, color, fs=9):
    r = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12",
                                 facecolor=color + "22", edgecolor=color, linewidth=1.5)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=fs, fontweight="bold", color=color)

def arrow(ax, x1, y1, x2, y2, label="", color="white"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.5))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.15, my+0.05, label, fontsize=7, color=COLORS["gray"], style="italic")

# Step 1: Create root
box(ax, 4.5, 13.0, 5.0, 0.6, "1. Create ROOT node (facts + empty history)", COLORS["blue"], 10)

# Step 2: Score root
box(ax, 4.5, 12.0, 5.0, 0.6, "2. Score root: closure_progress(root)", COLORS["blue"], 10)
arrow(ax, 7.0, 13.0, 7.0, 12.65)

# Step 3: frontier = [root]
box(ax, 4.5, 11.0, 5.0, 0.6, "3. FRONTIER = [root]", COLORS["blue"], 10)
arrow(ax, 7.0, 12.0, 7.0, 11.65)

# Step 4: while loop
box(ax, 3.5, 9.8, 7.0, 0.7, "4. WHILE frontier not empty AND budget > 0", COLORS["orange"], 10)
arrow(ax, 7.0, 11.0, 7.0, 10.55)

# Step 5: pop best
box(ax, 4.0, 8.8, 6.0, 0.6, "5. Sort frontier by reward. Pop BEST node.", COLORS["orange"], 10)
arrow(ax, 7.0, 9.8, 7.0, 9.45)

# Step 6: ask agent
box(ax, 4.0, 7.8, 6.0, 0.6, "6. Agent proposes k actions (or answers)", COLORS["green"], 10)
arrow(ax, 7.0, 8.8, 7.0, 8.45)

# Branch: Action vs Answer
box(ax, 0.5, 6.5, 4.5, 0.6, "Action → execute tool", COLORS["cyan"], 9)
box(ax, 8.5, 6.5, 5.0, 0.6, "Answer → verify()", COLORS["purple"], 9)
arrow(ax, 5.0, 7.8, 2.75, 7.15, "Action")
arrow(ax, 9.0, 7.8, 11.0, 7.15, "Answer")

# Action path
box(ax, 0.3, 5.5, 4.9, 0.6, "run_tool() → Observation", COLORS["cyan"], 9)
arrow(ax, 2.75, 6.5, 2.75, 6.15)
box(ax, 0.3, 4.5, 4.9, 0.6, "Create child node (new facts + history)", COLORS["cyan"], 9)
arrow(ax, 2.75, 5.5, 2.75, 5.15)
box(ax, 0.3, 3.5, 4.9, 0.6, "Score child: closure_progress()", COLORS["cyan"], 9)
arrow(ax, 2.75, 4.5, 2.75, 4.15)

# Action: reward check
box(ax, 0.3, 2.4, 4.9, 0.7, "reward ≥ 1.0?\nYes → derive answer → verify", COLORS["green"], 8)
arrow(ax, 2.75, 3.5, 2.75, 3.15)
box(ax, 0.3, 1.4, 4.9, 0.6, "reward > 0 → add to frontier", COLORS["blue"], 9)
arrow(ax, 2.75, 2.4, 2.75, 2.05)

# Answer path
box(ax, 8.3, 5.5, 5.2, 0.6, "Tier A → RETURN (verified!)", COLORS["green"], 9)
arrow(ax, 11.0, 6.5, 11.0, 6.15)
box(ax, 8.3, 4.5, 5.2, 0.6, "Tier B → save as best fallback", COLORS["orange"], 9)
arrow(ax, 11.0, 5.5, 11.0, 5.15)
box(ax, 8.3, 3.5, 5.2, 0.6, "Rejected + no prior reflection?", COLORS["red"], 9)
arrow(ax, 11.0, 4.5, 11.0, 4.15)
box(ax, 8.3, 2.4, 5.2, 0.7, "explain() → reflection text\nRe-queue node with reflection", COLORS["red"], 8)
arrow(ax, 11.0, 3.5, 11.0, 3.15)

# Loop back arrow
ax.annotate("", xy=(7.0, 9.85), xytext=(2.75, 1.4),
            arrowprops=dict(arrowstyle="-|>", color=COLORS["gray"], lw=1.2,
                           connectionstyle="arc3,rad=-0.3"))
ax.text(1.0, 0.8, "← loop back to step 4", fontsize=8, color=COLORS["gray"], style="italic")

# Final: exhausted
box(ax, 4.0, 0.2, 6.0, 0.7, "Budget exhausted → return best_tier_b or ABSTAIN", COLORS["gray"], 9)

plt.tight_layout()
plt.show()

In [ ]:
# Demo: the reflect/explain loop in action (using real oracle facts)
from src.contracts import TreeNode, Query
from src.engine.verifier import verify, explain
from src.agent.prompt import build_prompt

# Negation query: agent answers without fetching the exclusion list first
query = Query(type="negation", target="Pneumothorax", constraints={},
              raw_question="Are the lungs free of Pneumothorax?",
              parse_confidence=1.0, parser_tier="rule")

# Step 1: agent jumps to an answer without calling get_exclusion_list
node = TreeNode(state_facts=list(real_facts), history=[])
child = node.model_copy(update={"answer": "Yes, no Pneumothorax"})
result = verify(child, query, dag)
print("Step 1 \u2014 Agent answers prematurely")
print(f'  Agent answered: "Yes, no Pneumothorax"')
print(f"  Verifier says:  tier={result.tier}  (not Tier A \u2014 rejected!)")
print()

# Step 2: explain() diagnoses the failure
reflection = explain(child, query, dag)
print("Step 2 \u2014 explain() diagnoses the failure")
print(f"  {reflection}")
print()

# Step 3: the retry node carries this reflection into the agent's next prompt
retry_node = node.model_copy(update={"reflection": reflection})
prompt_before = build_prompt(node, query, k=3)
prompt_after = build_prompt(retry_node, query, k=3)

print("Step 3 \u2014 Reflection appears in the retry prompt")
print()
print("  Original prompt:  (no reflection line)")
assert "Reflection" not in prompt_before
print()
print("  Retry prompt:")
for line in prompt_after.split("\n"):
    if "Reflection" in line:
        print(f"  >>> {line}")
print()
print("The frozen LLM now sees WHY its answer failed and can adjust its strategy.")


In [ ]:
# Run a FULL pipeline trace for "Is there Cardiomegaly?" with MedGemmaAgent
# and print every step the tree search takes
from src.pipeline import run_with_facts
from src.agent.medgemma import MedGemmaAgent
from src.retrieval.index import RagIndex
from src.retrieval.retriever import Retriever
from src.retrieval.encoder import load_encoder
from PIL import Image

question = "Is there Cardiomegaly?"
print(f"Question: \"{question}\"")
print(f"Facts:    {[f.concept for f in real_facts]}")
print("=" * 70)

# We'll use run_with_facts and then trace the internals manually
# to show what happens step by step
from src.search.tree_search import search, _derive_answer, DEFAULT_IMG_WH
from src.tools.dispatch import run_tool

query = parser.parse(question)
print(f"\nParsed Query: type={query.type}, target={query.target}\n")

image = Image.open(IMAGE_PATH).convert("RGB")
agent = MedGemmaAgent(model_path="weights/medgemma-4b-it", quantize=True, load_model=True, image=image)

# Initialize retriever
with open("data/rag/vindr_cases.jsonl", "r", encoding="utf-8") as f:
    cases = [json.loads(line) for line in f]
index = RagIndex.load("data/rag/vindr_index.faiss", cases)
encoder = load_encoder(model_path="weights/BiomedCLIP", device="cuda")
retriever = Retriever(index=index, encoder=encoder)
retriever.set_query_emb(encoder.encode(image))

root = TreeNode(state_facts=list(real_facts), history=[])
root = root.model_copy(update={"reward": closure_progress(root, query, dag)})
print(f"Step 0: ROOT created, reward={root.reward:.2f}")

frontier = [root]
step = 0
vlm_fn = lambda crop, prompt: agent.generate(prompt, crop)

while frontier and step < 10:
    frontier.sort(key=lambda n: n.reward, reverse=True)
    node = frontier.pop(0)
    step += 1
    print(f"\nStep {step}: Expanding node (reward={node.reward:.2f}, history_len={len(node.history)})")

    proposals = agent.propose_actions(node, query, k=3)
    for p in proposals:
        if isinstance(p, str):
            print(f"  Agent proposes ANSWER: \"{p}\"")
            child = node.model_copy(update={"answer": p, "parent_id": id(node)})
            result = verify(child, query, dag)
            print(f"  Verifier: tier={result.tier}, answer='{result.answer}'")
            if result.tier == "A":
                print(f"\n{'='*70}")
                print(f"FINAL RESULT: tier=A, answer='{result.answer}', conf={result.conf:.2f}")
                print(f"Reasoning trace ({len(result.path)} steps):")
                for act, obs in result.path:
                    print(f"  → {act.tool}({act.args}) = {obs.result} (ok={obs.ok})")
                frontier = []  # stop
                break
        else:
            print(f"  Agent proposes ACTION: {p.tool}({p.args})")
            obs = run_tool(p, node.state_facts, dag, DEFAULT_IMG_WH, image=image, retriever=retriever, vlm_fn=vlm_fn)
            print(f"    → result={obs.result}, ok={obs.ok}")
            new_history = list(node.history) + [(p, obs)]
            child = TreeNode(state_facts=list(node.state_facts), history=new_history)
            child = child.model_copy(update={"reward": closure_progress(child, query, dag)})
            print(f"    → child reward={child.reward:.2f}")
            if child.reward >= 1.0:
                answer = _derive_answer(child, query)
                child = child.model_copy(update={"answer": answer})
                result = verify(child, query, dag)
                print(f"    → reward≥1.0! Derived answer='{answer}', verify tier={result.tier}")
                if result.tier == "A":
                    print(f"\n{'='*70}")
                    print(f"FINAL RESULT: tier=A, answer='{result.answer}', conf={result.conf:.2f}")
                    print(f"Reasoning trace ({len(result.path)} steps):")
                    for act, obs in result.path:
                        print(f"  → {act.tool}({act.args}) = {obs.result} (ok={obs.ok})")
                    frontier = []
                    break
            if child.reward > 0:
                frontier.append(child)


In [ ]:
# Run the full pipeline for multiple question types and visualize results
from src.pipeline import run_with_facts
from src.agent.medgemma import MedGemmaAgent
from src.retrieval.index import RagIndex
from src.retrieval.retriever import Retriever
from src.retrieval.encoder import load_encoder
from PIL import Image

image = Image.open(IMAGE_PATH).convert("RGB")
agent = MedGemmaAgent(model_path="weights/medgemma-4b-it", quantize=True, load_model=True, image=image)

# Load retriever
with open("data/rag/vindr_cases.jsonl", "r", encoding="utf-8") as f:
    cases = [json.loads(line) for line in f]
index = RagIndex.load("data/rag/vindr_index.faiss", cases)
encoder = load_encoder(model_path="weights/BiomedCLIP", device="cuda")
retriever = Retriever(index=index, encoder=encoder)
retriever.set_query_emb(encoder.encode(image))

test_cases = [
    ("Is there Cardiomegaly?",                    real_facts),
    ("Is there Pneumothorax?",                    real_facts),
    ("Are the lungs clear of Consolidation?",     real_facts),
    ("Where is the Pleural effusion?",            real_facts),
    ("How many findings are there?",              real_facts),
    ("What abnormality is visible?",              real_facts),
]

results = []
for q, facts in test_cases:
    result = run_with_facts(q, facts, dag, agent=agent, retriever=retriever, budget=20, k=3)
    results.append((q, result))

# Visualize results as a table
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis("off")
ax.set_title("Pipeline Results — Multiple Question Types", fontsize=14, fontweight="bold", pad=15)

table_data = []
for q, r in results:
    tier_color = {"A": COLORS["green"], "B": COLORS["orange"], "ABSTAIN": COLORS["red"]}[r.tier]
    table_data.append([q, r.tier, r.answer or "(empty)", f"{r.conf:.2f}", f"{len(r.path)} steps"])

col_labels = ["Question", "Tier", "Answer", "Conf", "Trace"]
table = ax.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)

for i in range(len(col_labels)):
    table[0, i].set_facecolor(COLORS["blue"] + "44")
    table[0, i].set_text_props(fontweight="bold", color="white")

for row in range(1, len(table_data) + 1):
    tier = table_data[row-1][1]
    color = {"A": COLORS["green"], "B": COLORS["orange"], "ABSTAIN": COLORS["red"]}[tier]
    table[row, 1].set_facecolor(color + "33")
    for col in range(len(col_labels)):
        table[row, col].set_text_props(color="white")
        table[row, col].set_facecolor("#1a1a2e" if col != 1 else color + "33")
        table[row, col].set_edgecolor(COLORS["gray"])

plt.tight_layout()
plt.show()


---
## 9. Multi-Hop Evaluation — Shared-Cause QA

**Files:** `src/eval/predictors.py` · `src/eval/multihop_metrics.py`

The multi-hop evaluation tests whether the system can answer: **"Can a single underlying condition cause both Finding A and Finding B?"**

### Five predictor systems compared:

| System | How it works | Grounded? |
|--------|-------------|-----------|
| **KG Oracle** | Directly queries `common_causes(a, b)` | Always |
| **Zero-shot** | MedGemma answers directly, no tools | Never |
| **CoT** | MedGemma chain-of-thought, no tools | Never |
| **ReAct** | MedGemma sees causal neighborhoods, answers freely | Only if cause is real |
| **KRONOS** | Model proposes → KG verifies → reflect → graph fallback | Always (by design) |

### KRONOS predictor flow:

```
1. MedGemma proposes a cause (zero-shot)
2. KG verification: dag.causal_edge(cause, a) AND dag.causal_edge(cause, b)?
   ├── YES → accept (verified trace)
   └── NO → reflection: tell model it was wrong, ask again
        ├── New candidate passes? → accept
        └── Still fails? → graph fallback: enumerate causal_neighbors(a)
             └── Any neighbor also causes b? → accept
                  └── Nothing? → answer "No"
```

### Five grading metrics:

| Metric | What it measures |
|--------|-----------------|
| **binary_accuracy** | Is the Yes/No correct? (all items) |
| **name_accuracy** | Is the named cause in the gold set? (gold-Yes items) |
| **grounding_rate** | Does the trace have real KG edges? (predicted-Yes items) |
| **hallucination_rate** | Is the cause NOT in the gold set? (predicted-Yes items) |
| **load_bearing_rate** | Does deleting support edges flip the answer? (single-cause Yes items) |

In [ ]:
# Demo: run the multi-hop predictors on sample items
from src.eval.predictors import (
    predict_mock, predict_zero_shot, predict_cot, predict_kronos, predict_react
)
from src.eval.multihop_metrics import grade_item, deletion_holds
from src.agent.medgemma import MedGemmaAgent
from PIL import Image

image = Image.open(IMAGE_PATH).convert("RGB")
agent = MedGemmaAgent(model_path="weights/medgemma-4b-it", quantize=True, load_model=True, image=image)
gen_fn = agent.generate

sample_items = [
    {
        "id": "demo_001",
        "finding_a": "Cardiomegaly",
        "finding_b": "Pleural effusion",
        "question": "Can a single condition cause both Cardiomegaly and Pleural effusion?",
        "answer": "Yes",
        "gold_causes": ["sarcoidosis", "amyloidosis", "lymphoma"],
        "support_edges": [["sarcoidosis", "Cardiomegaly"], ["sarcoidosis", "Pleural effusion"]],
        "single_cause": False,
    },
    {
        "id": "demo_002",
        "finding_a": "Pneumothorax",
        "finding_b": "Pleural thickening",
        "question": "Can a single condition cause both Pneumothorax and Pleural thickening?",
        "answer": "Yes",
        "gold_causes": ["tuberculosis", "trauma"],
        "support_edges": [["tuberculosis", "Pneumothorax"], ["tuberculosis", "Pleural thickening"]],
        "single_cause": False,
    },
]

predictors = [
    ("KG Oracle", lambda item: predict_mock(item, dag)),
    ("Zero-shot", lambda item: predict_zero_shot(item, gen_fn, image)),
    ("CoT",       lambda item: predict_cot(item, gen_fn, image)),
    ("ReAct",     lambda item: predict_react(item, dag, gen_fn, image)),
    ("KRONOS",    lambda item: predict_kronos(item, dag, gen_fn, image)),
]

for name, pred_fn in predictors:
    print(f"=== {name} Predictor ===\n")
    for item in sample_items:
        pred = pred_fn(item)
        grade = grade_item(item, pred, dag)
        print(f"  Q: {item['question']}")
        print(f"  Prediction: answer={pred['answer']}, cause={pred['cause']}")
        print(f"  Trace: {pred['trace']}")
        print(f"  Grade: binary_correct={grade['binary_correct']}, "
              f"name_correct={grade['name_correct']}, grounded={grade['grounded']}")
        print()

# Visualize the shared-cause graph for one example
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_title("Shared-Cause Example: Cardiomegaly + Pleural effusion", fontsize=13, fontweight="bold", pad=12)

a, b = "Cardiomegaly", "Pleural effusion"
shared = dag.common_causes(a, b)[:6]

G = nx.DiGraph()
G.add_node(a, role="finding")
G.add_node(b, role="finding")
for c in shared:
    G.add_node(c, role="cause")
    G.add_edge(c, a)
    G.add_edge(c, b)

pos = nx.spring_layout(G, k=2.0, seed=7)
node_c = [COLORS["red"] if G.nodes[n]["role"] == "finding" else COLORS["blue"] for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_c, node_size=2500, alpha=0.85)
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=COLORS["gray"], arrows=True,
                       arrowstyle="-|>", arrowsize=15, width=1.5, alpha=0.6)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_weight="bold", font_color="white")

legend_items = [
    mpatches.Patch(color=COLORS["red"], label="VinDr findings"),
    mpatches.Patch(color=COLORS["blue"], label="Shared causes (disorders)"),
]
ax.legend(handles=legend_items, loc="lower left", fontsize=9, framealpha=0.3)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"All shared causes of ({a}, {b}): {shared}")


---
## 10. Causal KG Statistics

A look at the scale and structure of the RGO-derived causal knowledge graph that powers multi-hop reasoning.

In [ ]:
# Causal KG statistics
causal = dag.causal
n_nodes = causal.number_of_nodes()
n_edges = causal.number_of_edges()

roles = {}
for _, d in causal.nodes(data=True):
    r = d.get("role", "unknown")
    roles[r] = roles.get(r, 0) + 1

# Degree distribution
in_degrees = [causal.in_degree(n) for n in causal.nodes()]
out_degrees = [causal.out_degree(n) for n in causal.nodes()]

# Seed finding connectivity
seeds = list(dag._seed_to_rgo.keys())
seed_in = []
for s in seeds:
    node = dag._seed_to_rgo[s]
    seed_in.append((s, causal.in_degree(node)))
seed_in.sort(key=lambda x: x[1], reverse=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Causal KG (RGO may_cause) — Statistics", fontsize=14, fontweight="bold", y=1.02)

# Plot 1: Node role distribution
ax = axes[0]
ax.bar(roles.keys(), roles.values(), color=[COLORS["blue"], COLORS["orange"]], alpha=0.8,
       edgecolor="white", linewidth=0.5)
ax.set_title("Node Roles", fontsize=11, fontweight="bold")
ax.set_ylabel("Count")
for i, (k, v) in enumerate(roles.items()):
    ax.text(i, v + 1, str(v), ha="center", fontweight="bold", color="white")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Plot 2: Degree distribution
ax = axes[1]
ax.hist(in_degrees, bins=20, alpha=0.7, color=COLORS["blue"], label="In-degree (# causes)", edgecolor="white")
ax.hist(out_degrees, bins=20, alpha=0.5, color=COLORS["orange"], label="Out-degree (# effects)", edgecolor="white")
ax.set_title("Degree Distribution", fontsize=11, fontweight="bold")
ax.set_xlabel("Degree")
ax.set_ylabel("# Nodes")
ax.legend(fontsize=8, framealpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Plot 3: VinDr finding connectivity (how many disorders cause each)
ax = axes[2]
names = [s[0] for s in seed_in]
counts = [s[1] for s in seed_in]
bars = ax.barh(range(len(names)), counts, color=COLORS["green"], alpha=0.8, edgecolor="white", linewidth=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=8)
ax.set_xlabel("# Disorders that may cause it")
ax.set_title("VinDr Finding Connectivity", fontsize=11, fontweight="bold")
ax.invert_yaxis()
for i, v in enumerate(counts):
    ax.text(v + 0.5, i, str(v), va="center", fontsize=8, color="white")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Causal KG: {n_nodes} nodes, {n_edges} edges")
print(f"Node roles: {roles}")
print(f"Mapped VinDr seeds: {len(seeds)}/{14}")

---
## 11. Complete System Architecture

Putting it all together — the full KRONOS architecture with every component and data flow.

In [ ]:
# Full system architecture diagram
fig, ax = plt.subplots(figsize=(18, 14))
ax.set_xlim(0, 18)
ax.set_ylim(0, 14)
ax.axis("off")
ax.set_title("KRONOS — Complete System Architecture", fontsize=18, fontweight="bold", color="white", pad=15)

def sbox(x, y, w, h, text, color, fs=9):
    r = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12",
                                 facecolor=color + "22", edgecolor=color, linewidth=2)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha="center", va="center", fontsize=fs, fontweight="bold", color=color)

def sarrow(x1, y1, x2, y2, color="white", lw=1.5):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw))

# === INPUTS (top) ===
sbox(1, 12.5, 3, 1, "CXR Image", COLORS["gray"], 12)
sbox(5.5, 12.5, 3, 1, "Question", COLORS["gray"], 12)

# === PERCEPTION ===
sbox(0.5, 10.5, 4, 1.2, "PERCEPTION\nYOLOv12s / Oracle", COLORS["blue"], 10)
sarrow(2.5, 12.5, 2.5, 11.75, COLORS["blue"])

# === PARSER ===
sbox(5, 10.5, 4, 1.2, "PARSER\nRule-based + LLM fallback", COLORS["orange"], 10)
sarrow(7, 12.5, 7, 11.75, COLORS["orange"])

# === Data contracts ===
sbox(0.5, 9.0, 4, 0.7, "[PerceptualFact, ...]", COLORS["blue"], 9)
sarrow(2.5, 10.5, 2.5, 9.75, COLORS["blue"])
sbox(5, 9.0, 4, 0.7, "Query(type, target, ...)", COLORS["orange"], 9)
sarrow(7, 10.5, 7, 9.75, COLORS["orange"])

# === TREE SEARCH (big box) ===
search_rect = mpatches.FancyBboxPatch((0.3, 2.5), 12.4, 6.0, boxstyle="round,pad=0.2",
                                       facecolor=COLORS["green"] + "11", edgecolor=COLORS["green"],
                                       linewidth=2.5, linestyle="--")
ax.add_patch(search_rect)
ax.text(6.5, 8.2, "TREE SEARCH (best-first, budget=20)", fontsize=12, fontweight="bold",
        color=COLORS["green"], ha="center")

sarrow(2.5, 9.0, 3.5, 8.15, COLORS["blue"])
sarrow(7, 9.0, 6.5, 8.15, COLORS["orange"])

# Inside search: Agent
sbox(0.8, 6.5, 4.5, 1.0, "AGENT\nMockAgent / MedGemma 4B\n(proposes actions or answers)", COLORS["purple"], 8)

# Inside search: Tool Layer
sbox(0.8, 4.8, 4.5, 1.2, "TOOL LAYER\nSymbolic: is_a, disjoint, anatomy_of,\nlaterality, exclusion, neighbors, path\nVisual: inspect, re_detect, compare\nRAG: retrieve", COLORS["cyan"], 7)
sarrow(3, 6.5, 3, 6.05, COLORS["purple"])

# Inside search: Verifier
sbox(6.5, 4.8, 5.8, 1.2, "VERIFIER\nclosure_progress() → search value\nverify() → Tier A / B / ABSTAIN\nexplain() → reflection text", COLORS["red"], 8)

# Inside search: Frontier
sbox(6.5, 6.5, 5.8, 1.0, "FRONTIER\n(sorted by reward, best-first)", COLORS["orange"], 9)

# Arrows inside search
sarrow(5.3, 7.0, 6.5, 7.0, COLORS["gray"])  # agent → frontier
sarrow(3, 4.8, 3, 4.0)  # tool → (feeds back)
sarrow(5.3, 5.4, 6.5, 5.4, COLORS["gray"])  # tool results → verifier
sarrow(9.4, 6.5, 9.4, 6.05, COLORS["red"])  # frontier → verifier

# Reflection arrow (verifier → agent)
ax.annotate("", xy=(3, 7.55), xytext=(9.4, 7.55),
            arrowprops=dict(arrowstyle="-|>", color=COLORS["red"], lw=1.2,
                           connectionstyle="arc3,rad=0.3"))
ax.text(6.2, 7.9, "reflection", fontsize=7, color=COLORS["red"], style="italic", ha="center")

# === ONTOLOGY DAG (right side) ===
sbox(13.5, 6.5, 4, 2.5, "ONTOLOGY DAG\n\nis-a hierarchy\ndisjoint-with\nanatomy zones\nlaterality rules\ncausal KG (RGO)", COLORS["blue"], 8)
sarrow(12.7, 5.8, 13.5, 7.0, COLORS["blue"])  # tools → DAG

# === CONCEPT LINKER (right) ===
sbox(13.5, 10.5, 4, 1.2, "CONCEPT LINKER\nSynonym dict + LLM", COLORS["pink"], 9)
sarrow(9, 10.8, 13.5, 10.8, COLORS["pink"])

# === LLM FALLBACK (right) ===
sbox(13.5, 4.0, 4, 1.2, "LLM FALLBACK\nGroq / Llama 3.3 70B\n(parser + linker tier-2)", COLORS["purple"], 8)

# === RAG (right) ===
sbox(13.5, 2.0, 4, 1.2, "RAG INDEX\nBiomedCLIP encoder\nFAISS / BruteForce", COLORS["cyan"], 8)
sarrow(12.7, 5.0, 13.5, 2.8, COLORS["cyan"])

# === OUTPUT ===
sbox(3, 0.5, 7, 1.2, "SearchResult\nanswer='Yes'  tier=A  conf=0.92\npath=[is_a(cardio,cardiac_abn) → ...]", COLORS["green"], 9)
sarrow(6.5, 2.5, 6.5, 1.75, COLORS["green"])

# === EVAL (bottom right) ===
sbox(11, 0.5, 6.5, 1.2, "MULTI-HOP EVAL\n5 predictors × 5 metrics\nbinary_acc · name_acc · grounding\nhallucination · load-bearing", COLORS["orange"], 8)
sarrow(10, 1.1, 11, 1.1, COLORS["orange"])

plt.tight_layout()
plt.show()

---
## 12. File Map

| Path | Role |
|------|------|
| `src/pipeline.py` | Entry point: wires perception → parsing → search |
| `src/contracts.py` | All data types: PerceptualFact, Query, Action, Observation, TreeNode, SearchResult |
| `src/perception/detector.py` | YOLOv12s wrapper → PerceptualFact list |
| `src/perception/oracle.py` | Ground-truth facts from VinDr train.csv |
| `src/question/parser.py` | Rule-based + LLM fallback question parser |
| `src/linking/linker.py` | Synonym + LLM concept linker |
| `src/ontology/dag.py` | OntologyDAG: is-a, disjoint, anatomy, laterality, causal graph |
| `src/agent/base.py` | Agent protocol interface |
| `src/agent/mock.py` | Deterministic scripted agent for testing |
| `src/agent/medgemma.py` | Frozen MedGemma 4B agent (HuggingFace transformers) |
| `src/agent/prompt.py` | Prompt construction + output parsing |
| `src/tools/dispatch.py` | Routes actions to symbolic / visual / retrieval |
| `src/tools/symbolic.py` | 7 symbolic tools (DAG operations) |
| `src/tools/visual.py` | 3 visual tools (inspect, re_detect, compare) + fold_facts |
| `src/retrieval/encoder.py` | BiomedCLIP image encoder |
| `src/retrieval/index.py` | BruteForce + FAISS vector index |
| `src/retrieval/retriever.py` | Wraps index + cached query embedding |
| `src/retrieval/tool.py` | Retrieve tool → Observation |
| `src/search/tree_search.py` | Best-first search with verifier-as-value |
| `src/engine/verifier.py` | closure_progress (value fn) + verify (tier gate) + explain (reflection) |
| `src/llm/groq_client.py` | Groq API wrapper for LLM fallback |
| `src/eval/predictors.py` | 5 multi-hop predictors (mock, zero-shot, CoT, ReAct, KRONOS) |
| `src/eval/multihop_metrics.py` | Grading: binary accuracy, name accuracy, grounding, hallucination, load-bearing |
| `data/ontology/dag.yaml` | Ontology: 33 nodes, is-a / part-of / located-in / disjoint edges |
| `data/ontology/causal_kg.yaml` | RGO may_cause subgraph: ~160 nodes, ~400 edges |
| `data/ontology/synonyms.yaml` | Finding name → canonical name mappings |
| `data/ontology/anatomy_zones.yaml` | Anatomy zone bounding boxes (normalized) |
| `data/ontology/exclusion_lists.yaml` | Closed-world exclusion lists per finding |